In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
spark = SparkSession.builder \
    .appName("aggregation") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

#### Join Customer and Address

In [0]:
%sql
use catalog gizmobox;
use schema silver;

select * from customers;

In [0]:
%sql
use catalog gizmobox;
use schema silver;

select * from addresses;

In [0]:
%sql
use catalog gizmobox;
use schema silver;

select 
    c.customer_id,
    c.customer_name,
    c.email,
    c.date_of_birth,
    c.member_since,
    c.telephone,
    a.shipping_address_line_1,
    a.shipping_city,
    a.shipping_state,
    a.shipping_postcode,
    a.billing_address_line_1,
    a.billing_city,
    a.billing_state,
    a.billing_postcode
   
from customers c 
left join addresses a on c.customer_id = a.customer_id

In [0]:
%sql
create table if not exists gizmobox.gold.customer_address as
select 
    c.customer_id,
    c.customer_name,
    c.email,
    c.date_of_birth,
    c.member_since,
    c.telephone,
    a.shipping_address_line_1,
    a.shipping_city,
    a.shipping_state,
    a.shipping_postcode,
    a.billing_address_line_1,
    a.billing_city,
    a.billing_state,
    a.billing_postcode
   
from customers c 
left join addresses a on c.customer_id = a.customer_id

In [0]:
%sql
select * from gizmobox.gold.customer_address

#### Monthly Order Summary

In [0]:
%sql
select *
from gizmobox.silver.orders;

In [0]:
%sql
select
    date_format(transaction_timestamp,'yyyy-MM') order_month,
    customer_id,
    count (distinct order_id) as total_orders,
    sum(quantity) as total_items_bought,
    sum(total_amount*quantity) as total_amount_spent
from gizmobox.silver.orders
group by date_format(transaction_timestamp,'yyyy-MM'), customer_id

In [0]:
%sql
create table if not exists gizmobox.gold.monthly_order as
select
    date_format(transaction_timestamp,'yyyy-MM') order_month,
    customer_id,
    count (distinct order_id) as total_orders,
    sum(quantity) as total_items_bought,
    sum(total_amount*quantity) as total_amount_spent
from gizmobox.silver.orders
group by date_format(transaction_timestamp,'yyyy-MM'), customer_id
